# EEG Seizure Correlation & Coherence Pipeline

Thin orchestration notebook. All logic lives in the helper modules (`config`, `obe_io`, `matching`, `correlation`, `coherence`, `plotting`).

## 1. Imports & Configuration

Pull in the helper modules and every `config` constant, and define the figure save/display helpers used by the summary sections.

In [ ]:
%matplotlib inline
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta
from IPython.display import display

from config import *
from obe_io import read_data
from matching import load_seizure_metadata, match_obe_files
from correlation import (
    channel_indices,
    datetime_to_sample,
    compute_rolling_correlation,
    filter_by_baseline,
    offset_to_day_zero,
)
from coherence import compute_seizure_coherence
import plotting as P


def render_per_animal(results, out_dir, name_fn, build_fn):
    # Save + display one summary figure per Animal_ID group; returns the figures.
    os.makedirs(out_dir, exist_ok=True)
    figs = []
    for animal, group in results.groupby('Animal_ID'):
        fig = build_fn(group)
        fig.savefig(os.path.join(out_dir, name_fn(animal)))
        display(fig)
        plt.close(fig)
        figs.append(fig)
    return figs


def render_global(out_dir, filename, fig):
    # Save + display a single global summary figure.
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, filename))
    display(fig)
    plt.close(fig)

## 2. Load Seizure Metadata

Read the seizure spreadsheet and parse the start/end datetimes for every record.

In [2]:
seizure_df = load_seizure_metadata(CSV_PATH)
print(f"Loaded {len(seizure_df)} seizure records")
seizure_df.head()

Loaded 112 seizure records


,Animal_no,Recording_order,Seizure,Racine_Scale,Date_Start,Date_End,Time_Start,Time_End,SeizureStartDT,SeizureEndDT
0,3.0,3,1,1,6/18/2013,6/18/2013,1:13:24 PM,1:15:14 PM,2013-06-18 13:13:24,2013-06-18 13:15:14
1,3.0,3,2,1,6/18/2013,6/18/2013,2:14:51 PM,2:16:40 PM,2013-06-18 14:14:51,2013-06-18 14:16:40
2,3.0,3,3,1,6/18/2013,6/18/2013,2:56:16 PM,2:58:16 PM,2013-06-18 14:56:16,2013-06-18 14:58:16
3,3.0,3,4,2,6/18/2013,6/18/2013,3:24:22 PM,3:26:09 PM,2013-06-18 15:24:22,2013-06-18 15:26:09
4,3.0,3,5,1,6/18/2013,6/18/2013,3:46:38 PM,3:48:23 PM,2013-06-18 15:46:38,2013-06-18 15:48:23


## 3. Match OBE Files to Seizures

Pair every `.OBE` recording with the seizures that overlap it (1-hour DST offset applied to header times).

In [3]:
matched_results = match_obe_files(FOLDER_PATH, seizure_df, DST_OFFSET_HOURS)

Starting search in 90 files...
Conflict in MSO1CDP6RE.OBE: Found multiple Recording_orders [np.int64(5), np.int64(7)]. Skipping.
Conflict in MSO1CDP6UH.OBE: Found multiple Recording_orders [np.int64(3), np.int64(5)]. Skipping.
Conflict in MSO1CDP75B.OBE: Found multiple Recording_orders [np.int64(5), np.int64(7)]. Skipping.

--- Matching Complete ---
Successfully matched 87 OBE files.


## 4. Matched Seizure Summary

List the matched files with their animal, recording order, and seizure count.

In [4]:
total = sum(len(m.seizures) for m in matched_results)
print(f"{len(matched_results)} files matched, {total} seizures total (1hr offset applied)\n")
for m in matched_results:
    s = m.seizures
    print(f"{m.filename}: Animal {s['Animal_no'].iloc[0]}, "
          f"Rec {s['Recording_order'].iloc[0]}, {len(s)} seizure(s)")

87 files matched, 102 seizures total (1hr offset applied)

MSO1CDP6IE.OBE: Animal 3.0, Rec 3, 1 seizure(s)
MSO1CDP6IF.OBE: Animal 3.0, Rec 3, 2 seizure(s)
MSO1CDP6IG.OBE: Animal 3.0, Rec 3, 2 seizure(s)
MSO1CDP6IH.OBE: Animal 3.0, Rec 3, 4 seizure(s)
MSO1CDP6IL.OBE: Animal 3.0, Rec 3, 1 seizure(s)
MSO1CDP6IM.OBE: Animal 3.0, Rec 3, 1 seizure(s)
MSO1CDP6J0.OBE: Animal 3.0, Rec 3, 1 seizure(s)
MSO1CDP6J3.OBE: Animal 3.0, Rec 3, 1 seizure(s)
MSO1CDP6J4.OBE: Animal 3.0, Rec 3, 1 seizure(s)
MSO1CDP6K2.OBE: Animal 5.0, Rec 5, 1 seizure(s)
MSO1CDP6K9.OBE: Animal 5.0, Rec 5, 2 seizure(s)
MSO1CDP6KA.OBE: Animal 5.0, Rec 5, 2 seizure(s)
MSO1CDP6KB.OBE: Animal 5.0, Rec 5, 1 seizure(s)
MSO1CDP6KD.OBE: Animal 5.0, Rec 5, 1 seizure(s)
MSO1CDP6KM.OBE: Animal 5.0, Rec 5, 1 seizure(s)
MSO1CDP6QA.OBE: Animal 3.1, Rec 3, 1 seizure(s)
MSO1CDP6QG.OBE: Animal 3.1, Rec 3, 2 seizure(s)
MSO1CDP6QH.OBE: Animal 5.0, Rec 5, 1 seizure(s)
MSO1CDP6RF.OBE: Animal 7.0, Rec 7, 4 seizure(s)
MSO1CDP6RG.OBE: Animal 7.0, R

## 5. 60-Minute EEG Overview Plots

Read each matched file's two channels and render a 60-minute decimated overview with seizure markers.

In [5]:
os.makedirs(OVERVIEW_OUTPUT_DIR, exist_ok=True)
overview_plots = []

for m in matched_results:
    row = m.seizures.iloc[0]
    ch1, ch2 = channel_indices(row['Recording_order'], CH1_OFFSET, CH2_OFFSET, CHANNEL_STRIDE)
    path = os.path.join(FOLDER_PATH, m.filename)
    d1, d2 = read_data(path, ch1), read_data(path, ch2)
    fig = P.plot_overview(d1, d2, m.seizures, m.corrected_start, m.header.samp_freq,
                          ch1, ch2, m.filename, row['Animal_no'], row['Recording_order'],
                          DOWNSAMPLE_FACTOR, NUM_MINS_PER_SUBPLOT, OVERVIEW_NUM_SUBPLOTS)
    safe = re.sub(r'[\\/:*?"<>|]', '_', m.filename)
    fig.savefig(os.path.join(OVERVIEW_OUTPUT_DIR,
                f"Animal{row['Animal_no']}_Rec{row['Recording_order']}_{safe}.png"))
    overview_plots.append(fig)
    plt.close(fig)

print(f"{len(overview_plots)} overview plots generated.")

87 overview plots generated.


In [ ]:
for fig in overview_plots:
    display(fig)

## 6. Baseline Correlation Pre-Filter

Drop seizures whose pre-seizure baseline is already too correlated (or whose window is out of bounds) before the main analysis.

In [ ]:
seizures_to_analyze, excluded_seizures = filter_by_baseline(
    matched_results, FOLDER_PATH,
    baseline_duration_secs=BASELINE_DURATION_SECS,
    baseline_corr_threshold=BASELINE_CORR_THRESHOLD,
    rolling_window_secs=ROLLING_WINDOW_SECS,
    rolling_step_secs=ROLLING_STEP_SECS,
    ch1_offset=CH1_OFFSET, ch2_offset=CH2_OFFSET, channel_stride=CHANNEL_STRIDE,
)

## 7. Main Analysis Loop

For each surviving seizure: rolling correlation over a padded window, plus during-vs-pre-seizure coherence and phase delay. Results collected into two sibling DataFrames.

In [ ]:
os.makedirs(CORRELATION_OUTPUT_DIR, exist_ok=True)
os.makedirs(COHERENCE_OUTPUT_DIR, exist_ok=True)

correlation_rows, coherence_rows = [], []
correlation_plots, coherence_plots = [], []
coherence_details = []

for n, (match_idx, row_idx, baseline_corr) in enumerate(seizures_to_analyze, start=1):
    m = matched_results[match_idx]
    row = m.seizures.iloc[row_idx]
    path = os.path.join(FOLDER_PATH, m.filename)
    sf = m.header.samp_freq
    ch1, ch2 = channel_indices(row['Recording_order'], CH1_OFFSET, CH2_OFFSET, CHANNEL_STRIDE)
    print(f"[{n}] Seizure {row['Seizure']} (Animal {row['Animal_no']})")

    try:
        d1_full, d2_full = read_data(path, ch1), read_data(path, ch2)

        # Rolling correlation over a window centred on the seizure.
        midpoint = row['SeizureStartDT'] + (row['SeizureEndDT'] - row['SeizureStartDT']) / 2
        pad_start = midpoint - timedelta(seconds=PADDING_WINDOW_SECS)
        pad_end = midpoint + timedelta(seconds=PADDING_WINDOW_SECS)
        start_samp = datetime_to_sample(pad_start, m.corrected_start, sf)
        end_samp = datetime_to_sample(pad_end, m.corrected_start, sf)
        if start_samp < 0 or end_samp > m.header.samples:
            print(f"  ! Seizure {row['Seizure']} window out of bounds. Skipping.")
            continue

        data1, data2 = d1_full[start_samp:end_samp], d2_full[start_samp:end_samp]
        corrs, time_axis = compute_rolling_correlation(
            data1, data2, sf, ROLLING_WINDOW_SECS, ROLLING_STEP_SECS)

        sz_start_sec = (row['SeizureStartDT'] - pad_start).total_seconds()
        sz_end_sec = (row['SeizureEndDT'] - pad_start).total_seconds()
        seiz_corrs = corrs[(time_axis >= sz_start_sec) & (time_axis <= sz_end_sec)]
        seconds_above = int(np.sum(np.abs(seiz_corrs) > CORR_THRESHOLD)) * ROLLING_STEP_SECS
        avg_corr = float(np.mean(np.abs(seiz_corrs))) if seiz_corrs.size else 0.0
        seiz_dur = (row['SeizureEndDT'] - row['SeizureStartDT']).total_seconds()

        correlation_rows.append(dict(
            OBE_File=m.filename, File_Start_DT=m.corrected_start, Animal_ID=row['Animal_no'],
            Racine_Scale=int(row['Racine_Scale']), Seconds_Above_Threshold=seconds_above,
            Seizure_Duration_Sec=float(seiz_dur), Avg_Corr_Strength=avg_corr, Ch1=ch1, Ch2=ch2))

        # Zoomed raw seizure segments + assembled correlation figure.
        ss = datetime_to_sample(row['SeizureStartDT'], pad_start, sf)
        se = datetime_to_sample(row['SeizureEndDT'], pad_start, sf)
        raw1, raw2 = data1[ss:se], data2[ss:se]
        base_str = 'N/A' if np.isnan(baseline_corr) else round(baseline_corr, 2)
        title = f"Animal {row['Animal_no']} | Sec Above: {round(seconds_above, 1)}s | Base Corr: {base_str}"
        date_str = row['SeizureStartDT'].strftime('%Y-%m-%d')
        fig = P.assemble_correlation_figure(
            corrs, time_axis, sz_start_sec, sz_end_sec, raw1, raw2, sf,
            CORR_THRESHOLD, SEGMENT_SECS, title=title)
        fig.savefig(os.path.join(CORRELATION_OUTPUT_DIR,
                    f"Animal_{row['Animal_no']}_Seiz_{row['Seizure']}_{date_str}.png"))
        correlation_plots.append(fig)
        plt.close(fig)

        # Coherence: during seizure vs pre-seizure baseline (band-averaged).
        pre_start_s = datetime_to_sample(
            row['SeizureStartDT'] - timedelta(seconds=PRE_SEIZURE_SECS), m.corrected_start, sf)
        sz_start_s = datetime_to_sample(row['SeizureStartDT'], m.corrected_start, sf)
        sz_end_s = datetime_to_sample(row['SeizureEndDT'], m.corrected_start, sf)
        if pre_start_s >= 0 and sz_end_s <= m.header.samples:
            coh = compute_seizure_coherence(
                d1_full, d2_full, sz_start_s, sz_end_s, pre_start_s, sz_start_s, sf,
                COH_WINDOW, COH_NOVERLAP, COH_NFFT, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ,
                phase_msc_threshold=PHASE_MSC_THRESHOLD,)
            coherence_rows.append(dict(
                OBE_File=m.filename,
                Animal_ID=row['Animal_no'],
                Seizure=int(row['Seizure']),
                File_Start_DT=m.corrected_start,
                Avg_MSC_Seizure_Low=coh['avg_msc_seizure_low'],
                Avg_MSC_Pre_Low=coh['avg_msc_pre_low'],
                Avg_MSC_Seizure_High=coh['avg_msc_seizure_high'],
                Avg_MSC_Pre_High=coh['avg_msc_pre_high'],
                Tau_ms_Seizure=coh['tau_ms_seizure'],
                Tau_ms_Pre=coh['tau_ms_pre'],
                Phase_R2_Seizure=coh['phase_r2_seizure'],
                Phase_R2_Pre=coh['phase_r2_pre'],
                Phase_Nbins_Seizure=coh['phase_nbins_seizure'],
                Phase_Nbins_Pre=coh['phase_nbins_pre'],))
            coherence_details.append({**coh,
                "Animal_ID": row['Animal_no'],
                "Seizure": int(row['Seizure'])})
            cfig = P.plot_coherence(
                coh['freqs_pre'], coh['msc_pre'], coh['freqs_seizure'], coh['msc_seizure'], sf,
                pre_seizure_secs=PRE_SEIZURE_SECS,
                title=f"Coherence | Animal {row['Animal_no']} Seiz {row['Seizure']}")
            cfig.savefig(os.path.join(COHERENCE_OUTPUT_DIR,
                         f"Coherence_Animal_{row['Animal_no']}_Seiz_{row['Seizure']}_{date_str}.png"))
            coherence_plots.append(cfig)
            plt.close(cfig)
        else:
            print(f"  ! Coherence skipped for Seizure {row['Seizure']}: pre-window out of bounds.")

    except Exception as e:
        print(f"  ERROR on Seizure {row['Seizure']} in {m.filename}: {e}")

correlation_results = pd.DataFrame(correlation_rows).sort_values('Animal_ID').reset_index(drop=True)
coherence_results = pd.DataFrame(coherence_rows).reset_index(drop=True)
print(f"\nAnalysis complete: {len(correlation_results)} correlation rows, "
      f"{len(coherence_results)} coherence rows.")

## 8. Correlation Plots

Per-seizure rolling-correlation traces with their zoomed raw seizure segments.

In [ ]:
for fig in correlation_plots:
    display(fig)

## 9. Coherence Plots

Per-seizure pre- vs during-seizure magnitude-squared coherence across frequency.

In [ ]:
for fig in coherence_plots:
    display(fig)

## 10. Phase Delay Exploration

Per-animal raw MSC and phase-vs-frequency grids, plus an interactive box plot of the phase-derived inter-channel delays. Reloads `plotting` first so edits to the module take effect.

In [ ]:
import importlib
import plotting
P = importlib.reload(plotting)

In [ ]:
import os
from ipywidgets import interact, FloatSlider

os.makedirs(COHERENCE_DELAY_OUTPUT_DIR, exist_ok=True)

#top: raw MSC per animal
msc_grid = P.plot_msc_by_animal(coherence_details, band_low=COH_BAND_LOW_HZ, which="seizure")
msc_grid.savefig(os.path.join(COHERENCE_DELAY_OUTPUT_DIR, "MSC_By_Animal.png"))
display(msc_grid); plt.close(msc_grid)

#middle: phase per animal (opacity = MSC)
grid = P.plot_phase_by_animal(coherence_details, band_low=COH_BAND_LOW_HZ,
                              which="seizure", color_by="msc", phase_ylim=(-3, 3))
grid.savefig(os.path.join(COHERENCE_DELAY_OUTPUT_DIR, "Phase_By_Animal_MSCopacity.png"))
display(grid); plt.close(grid)

#bottom: interactive box plot
@interact(r2_min=FloatSlider(min=0.0, max=1, step=0.01, value=0.5,
                             continuous_update=True))
def _show(r2_min):
    fig = P.plot_delay_distribution(coherence_results, r2_min=r2_min, nbins_min=3)
    plt.show()

## 11. Results Tables

The collected per-seizure correlation and coherence DataFrames.

In [ ]:
display(correlation_results)

In [ ]:
display(coherence_results)

## 12. Per-Animal Correlation Summaries

Longitudinal correlation trends per animal: duration, time above threshold, strength, and Racine severity.

In [ ]:
render_per_animal(
    correlation_results, CORRELATION_SUMMARY_OUTPUT_DIR,
    lambda a: f"Animal_{a}_Summary.png",
    lambda g: P.plot_animal_summary(g, CORR_THRESHOLD, COLORMAP, RACINE_SCALE_MAX),
)

## 13. Per-Animal Coherence Summaries

Per-animal band-averaged coherence over time, plus the seizure/pre-seizure MSC ratio.

In [ ]:
render_per_animal(
    coherence_results, COHERENCE_SUMMARY_OUTPUT_DIR,
    lambda a: f"Animal_{a}_Coherence_Summary.png",
    lambda g: P.plot_coherence_animal_summary(g, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ,
                                              pre_seizure_secs=PRE_SEIZURE_SECS),
)

# Seizure/pre-seizure MSC ratio over time, beneath the coherence summary.
render_per_animal(
    coherence_results, COHERENCE_SUMMARY_OUTPUT_DIR,
    lambda a: f"Animal_{a}_Coherence_Ratio.png",
    lambda g: P.plot_coherence_ratio_animal_summary(g, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ),
)

## 14. Per-Animal Phase Delay Summaries

For each animal, the phase-derived inter-channel time delay (ms) per seizure over time, with point colour = phase-slope fit R² and size = number of coherent bins behind each estimate. Saved to `Coherence_Delay_Output`.

In [ ]:
render_per_animal(
    coherence_results, COHERENCE_DELAY_OUTPUT_DIR,
    lambda a: f"Animal_{a}_Phase_Delay.png",
    lambda g: P.plot_phase_delay_animal_summary(g, COH_BAND_LOW_HZ,
                                               pre_seizure_secs=PRE_SEIZURE_SECS),
)

## 15. Global Correlation Summary (Offset to Day 0)

All animals overlaid on a common experimental-day axis, coloured by animal.

In [ ]:
norm_corr = offset_to_day_zero(correlation_results, day_zero=DAY_ZERO)
render_global(CORRELATION_SUMMARY_OUTPUT_DIR, "Global_Correlation_Summary.png",
              P.plot_global_summary(norm_corr, RACINE_SCALE_MAX, DAY_ZERO))

## 16. Global Coherence Summary (Offset to Day 0)

Band-averaged coherence and the seizure/pre-seizure ratio across all animals on the Day-0 axis.

In [ ]:
norm_coh = offset_to_day_zero(coherence_results, day_zero=DAY_ZERO)
render_global(COHERENCE_SUMMARY_OUTPUT_DIR, "Global_Coherence_Summary.png",
              P.plot_coherence_global_summary(norm_coh, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ,
                                              day_zero=DAY_ZERO))

In [ ]:
render_global(COHERENCE_SUMMARY_OUTPUT_DIR, "Global_Coherence_Ratio.png",
              P.plot_coherence_ratio_global_summary(norm_coh, COH_BAND_LOW_HZ, COH_BAND_HIGH_HZ,
                                                    day_zero=DAY_ZERO))

## 17. Global Phase Delay Summary (Offset to Day 0)

All animals overlaid on the experimental-day axis. One point per seizure (during-seizure delay): colour = phase-slope fit R², size = coherent bins used, and marker shape = animal.

In [ ]:
render_global(COHERENCE_DELAY_OUTPUT_DIR, "Global_Phase_Delay.png",
              P.plot_phase_delay_global_summary(norm_coh, COH_BAND_LOW_HZ, day_zero=DAY_ZERO))